# CIFAR Hydra on Google Colab

This notebook runs the MAEG3080 CIFAR-100 project on Colab with checkpoints stored in Google Drive.

Before running, switch the runtime to **GPU**.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from google.colab import files
import os
import pathlib
import shutil
import tarfile

WORK_ROOT = pathlib.Path('/content')
PROJECT_ROOT = WORK_ROOT / 'cifar-hydra'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/cifar_hydra_runs')
BUNDLE_NAME = 'cifar_hydra_colab_bundle.tar.gz'
BUNDLE_PATH = DRIVE_ROOT / BUNDLE_NAME

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if not BUNDLE_PATH.exists():
    print(f'Upload {BUNDLE_NAME} from your computer.')
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded:
        raise FileNotFoundError(f'Please upload {BUNDLE_NAME}.')
    with open(BUNDLE_PATH, 'wb') as handle:
        handle.write(uploaded[BUNDLE_NAME])

with tarfile.open(BUNDLE_PATH, 'r:gz') as archive:
    archive.extractall(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Drive root:', DRIVE_ROOT)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
import pathlib
import sys
import torch

DATA_DIR = DRIVE_ROOT / 'data'
SAVE_DIR = DRIVE_ROOT / 'checkpoints'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'

DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTHON_BIN'] = sys.executable
os.environ['DATA_DIR'] = str(DATA_DIR)
os.environ['SAVE_DIR'] = str(SAVE_DIR)
os.environ['DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['NUM_WORKERS'] = '2'
os.environ['BATCH_SIZE'] = '128'

print('DATA_DIR =', DATA_DIR)
print('SAVE_DIR =', SAVE_DIR)
print('DEVICE =', os.environ['DEVICE'])


## Run Experiments

Recommended order: `run-c`, `run-d`, `run-e`, `select-best`, then `run-f`.


In [ ]:
import os
import subprocess
import sys

run_mode = 'run-d'  # change to run-c, run-d, run-e, or run-f
best_run = 'hydra_run_d'  # only used for run-f

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
if run_mode == 'run-f':
    env['BEST_RUN'] = best_run

process = subprocess.Popen(
    ['bash', 'scripts/hydra_ladder.sh', run_mode],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)


In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        'scripts/select_best_run.py',
        os.path.join(os.environ['SAVE_DIR'], 'hydra_run_c'),
        os.path.join(os.environ['SAVE_DIR'], 'hydra_run_d'),
        os.path.join(os.environ['SAVE_DIR'], 'hydra_run_e'),
    ],
    check=True,
)


In [ ]:
import os
import pathlib
import subprocess
import sys

checkpoint_path = pathlib.Path(os.environ['SAVE_DIR']) / 'hydra_run_f_final' / 'best.pt'

subprocess.run(
    [
        sys.executable,
        'results.py',
        '--checkpoint',
        str(checkpoint_path),
        '--data-dir',
        os.environ['DATA_DIR'],
        '--batch-size',
        '256',
        '--num-workers',
        '2',
        '--device',
        os.environ['DEVICE'],
    ],
    check=True,
)
